# 03. 사용주기 AI 모델 비교 실험

이 노트북은 `02_Modeling_RF.ipynb`의 모델 교체 계약을 유지하면서 여러 회귀 모델을 같은 데이터, 같은 feature, 같은 split으로 비교한다.

목표는 서버 코드 수정 없이 `rf_final_model.pkl`, `model_meta.json`만 교체 가능한 최종 후보 모델을 찾는 것이다.

## 1. Import 및 경로 설정

In [ ]:
import json
import time
import warnings
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    precision_score,
    recall_score,
    f1_score,
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
TERM_MONTHS = 6
sns.set_theme(style="whitegrid")

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    """노트북 실행 위치가 root/notebooks 어디든 프로젝트 루트를 찾는다."""
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "dataset" / "create_data" / "data_ml" / "phase4_training_data.csv").exists():
            return path
    raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다. U-sto_AI 폴더에서 실행해 주세요.")

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "dataset" / "create_data" / "data_ml" / "phase4_training_data.csv"
EXPERIMENT_DIR = PROJECT_ROOT / "ai_model" / "experiments"
RUN_DIR = EXPERIMENT_DIR / "runs"
RESULT_PATH = EXPERIMENT_DIR / "model_comparison.csv"

for directory in [EXPERIMENT_DIR, RUN_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_PATH: {DATA_PATH}")

## 2. 데이터 로드 및 모델 계약 고정

In [ ]:
FEATURES = [
    "내용연수",
    "취득금액",
    "부서가혹도",
    "가격민감도",
    "장비중요도",
    "리드타임등급",
    "취득월",
    "G2B목록명_Code",
    "물품분류명_Code",
    "운용부서코드_Code",
    "캠퍼스_Code",
]
TARGET = "실제수명_개월"

df = pd.read_csv(DATA_PATH)
df["취득일자"] = pd.to_datetime(df["취득일자"], errors="coerce")
df[TARGET] = df["실제수명"] * 12

missing_features = [col for col in FEATURES if col not in df.columns]
if missing_features:
    raise ValueError(f"누락된 feature 컬럼: {missing_features}")

split_counts = df["데이터세트구분"].value_counts(dropna=False)
display(split_counts.to_frame("count"))
display(df[FEATURES + [TARGET, "운용연차", "데이터세트구분"]].head())

In [ ]:
train_mask = df["데이터세트구분"].isin(["Train", "Valid"])
test_mask = df["데이터세트구분"].eq("Test")
pred_mask = df["데이터세트구분"].eq("Prediction")

X_train = df.loc[train_mask, FEATURES]
y_train = df.loc[train_mask, TARGET]
X_test = df.loc[test_mask, FEATURES]
y_test = df.loc[test_mask, TARGET]
df_predict = df.loc[pred_mask].copy()

test_age_months = df.loc[test_mask, "운용연차"].to_numpy() * 12
actual_rul_test = y_test.to_numpy() - test_age_months
actual_term_fail_test = actual_rul_test <= TERM_MONTHS

print(f"Train rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Prediction rows: {len(df_predict):,}")

## 3. 후보 모델 정의

In [ ]:
def numeric_pipeline(model, scale: bool = False):
    steps = [SimpleImputer(strategy="median")]
    if scale:
        steps.append(StandardScaler())
    steps.append(model)
    return make_pipeline(*steps)

MODEL_REGISTRY = {
    "DummyMedian": numeric_pipeline(DummyRegressor(strategy="median")),
    "LinearRegression": numeric_pipeline(LinearRegression(), scale=True),
    "Ridge": numeric_pipeline(Ridge(alpha=1.0, random_state=RANDOM_STATE), scale=True),
    "ElasticNet": numeric_pipeline(
        ElasticNet(alpha=0.001, l1_ratio=0.2, random_state=RANDOM_STATE, max_iter=10_000),
        scale=True,
    ),
    "KNN": numeric_pipeline(KNeighborsRegressor(n_neighbors=7, weights="distance")),
    "DecisionTree": numeric_pipeline(
        DecisionTreeRegressor(max_depth=12, min_samples_leaf=2, random_state=RANDOM_STATE)
    ),
    "RandomForest": numeric_pipeline(
        RandomForestRegressor(
            n_estimators=500,
            max_depth=10,
            min_samples_split=5,
            max_features=1.0,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    ),
    "ExtraTrees": numeric_pipeline(
        ExtraTreesRegressor(
            n_estimators=500,
            max_depth=12,
            min_samples_leaf=1,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    ),
    "GradientBoosting": numeric_pipeline(GradientBoostingRegressor(random_state=RANDOM_STATE)),
    "HistGradientBoosting": numeric_pipeline(
        HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=RANDOM_STATE)
    ),
    "XGBoost": numeric_pipeline(
        XGBRegressor(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    ),
    "LightGBM": numeric_pipeline(
        LGBMRegressor(
            n_estimators=500,
            learning_rate=0.03,
            num_leaves=31,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=-1,
        )
    ),
    "CatBoost": numeric_pipeline(
        CatBoostRegressor(
            iterations=500,
            depth=6,
            learning_rate=0.03,
            loss_function="RMSE",
            random_seed=RANDOM_STATE,
            verbose=False,
        )
    ),
}

list(MODEL_REGISTRY.keys())

## 4. 공통 평가 함수

In [ ]:
def evaluate_predictions(model_name: str, y_true, y_pred, elapsed_sec: float) -> dict:
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))

    pred_rul = y_pred - test_age_months
    pred_term_fail = pred_rul <= TERM_MONTHS

    return {
        "model": model_name,
        "rmse_months": rmse,
        "mae_months": mae,
        "r2": r2,
        "term_precision": float(precision_score(actual_term_fail_test, pred_term_fail, zero_division=0)),
        "term_recall": float(recall_score(actual_term_fail_test, pred_term_fail, zero_division=0)),
        "term_f1": float(f1_score(actual_term_fail_test, pred_term_fail, zero_division=0)),
        "elapsed_sec": float(elapsed_sec),
    }

def extract_feature_importance(pipeline, feature_names: list[str]) -> pd.DataFrame | None:
    estimator = pipeline.steps[-1][1] if hasattr(pipeline, "steps") else pipeline
    if hasattr(estimator, "feature_importances_"):
        values = estimator.feature_importances_
    elif hasattr(estimator, "coef_"):
        values = np.ravel(estimator.coef_)
    else:
        return None
    return (
        pd.DataFrame({"feature": feature_names, "importance": values})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

## 5. 1차 벤치마크 실행

In [ ]:
benchmark_rows = []
trained_models = {}
test_predictions = {}

for model_name, model in MODEL_REGISTRY.items():
    print(f"[TRAIN] {model_name}")
    started = time.perf_counter()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    elapsed = time.perf_counter() - started

    metrics = evaluate_predictions(model_name, y_test.to_numpy(), y_pred, elapsed)
    benchmark_rows.append(metrics)
    trained_models[model_name] = model
    test_predictions[model_name] = y_pred

comparison_df = pd.DataFrame(benchmark_rows).sort_values("rmse_months").reset_index(drop=True)
comparison_df.to_csv(RESULT_PATH, index=False, encoding="utf-8-sig")
display(comparison_df)
print(f"비교표 저장 완료: {RESULT_PATH}")

In [ ]:
plt.figure(figsize=(11, 5))
sns.barplot(data=comparison_df, x="rmse_months", y="model", palette="viridis")
plt.title("Model Benchmark - RMSE(months)")
plt.xlabel("RMSE (months)")
plt.ylabel("Model")
plt.tight_layout()
plt.show()

## 6. 최상위 후보 산출물 저장

아래 셀은 1차 벤치마크에서 RMSE가 가장 낮은 모델을 별도 실험 폴더에 저장한다. 서버 배포 경로는 아직 덮어쓰지 않는다.

In [ ]:
best_row = comparison_df.iloc[0].to_dict()
best_model_name = best_row["model"]
best_model = trained_models[best_model_name]
best_pred = test_predictions[best_model_name]

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
safe_model_name = best_model_name.lower().replace(" ", "_")
best_run_dir = RUN_DIR / f"{run_id}_{safe_model_name}"
best_run_dir.mkdir(parents=True, exist_ok=True)

model_path = best_run_dir / "model.pkl"
meta_path = best_run_dir / "model_meta.json"
metrics_path = best_run_dir / "metrics.json"
pred_path = best_run_dir / "test_predictions.csv"
fi_path = best_run_dir / "feature_importance.png"

joblib.dump(best_model, model_path)

model_meta = {
    "run_id": run_id,
    "model_name": best_model_name,
    "target": TARGET,
    "features": FEATURES,
    "rmse_months": best_row["rmse_months"],
    "mae_months": best_row["mae_months"],
    "r2": best_row["r2"],
    "trained_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "data_path": str(DATA_PATH.relative_to(PROJECT_ROOT)),
    "split_rule": "Train+Valid train, Test evaluation, Prediction inference",
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(model_meta, f, ensure_ascii=False, indent=2)
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(best_row, f, ensure_ascii=False, indent=2)

pred_df = df.loc[test_mask, ["물품고유번호", "G2B목록명", "운용연차", "실제수명"]].copy()
pred_df["actual_total_life_months"] = y_test.to_numpy()
pred_df["pred_total_life_months"] = best_pred
pred_df["actual_rul_months"] = actual_rul_test
pred_df["pred_rul_months"] = best_pred - test_age_months
pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")

fi_df = extract_feature_importance(best_model, FEATURES)
if fi_df is not None:
    plt.figure(figsize=(9, 5))
    sns.barplot(data=fi_df, x="importance", y="feature", palette="mako")
    plt.title(f"Feature Importance - {best_model_name}")
    plt.tight_layout()
    plt.savefig(fi_path, dpi=150, bbox_inches="tight")
    plt.show()

print(f"Best model: {best_model_name}")
print(f"Run dir: {best_run_dir}")

## 7. 서버 호환 후처리 스모크 테스트

In [ ]:
def predict_assets(model, X_predict, current_age_years):
    predicted_total_months = model.predict(X_predict)
    current_age_months = current_age_years * 12
    raw_rul = predicted_total_months - current_age_months
    rng = np.random.default_rng(RANDOM_STATE)
    zombie_months = rng.uniform(1.0, 4.0, size=len(raw_rul))
    rul_months = np.where(raw_rul <= 0.5, zombie_months, raw_rul)
    return np.round(rul_months, 1)

def calculate_failure_date(rul_months, base_date):
    base_dt = pd.to_datetime(base_date)
    return base_dt + pd.to_timedelta(rul_months * 30.44, unit="D")

def assign_matrix_zone(rul_months, importance):
    if rul_months <= 6 and importance >= 0.7:
        return "즉시 조달"
    if rul_months <= 6 and importance < 0.7:
        return "일반 교체"
    if rul_months > 6 and importance >= 0.7:
        return "집중 모니터링"
    return "정상 운용"

def get_term_months(target_term):
    term_map = {"1학기": 3, "여름계절학기": 6, "2학기": 9, "겨울계절학기": 12}
    return term_map.get(target_term, 3)

def classify_term_failure(rul_value, term_months):
    return rul_value <= term_months

In [ ]:
smoke_df = df_predict.copy()
loaded_model = joblib.load(model_path)
with open(meta_path, "r", encoding="utf-8") as f:
    loaded_meta = json.load(f)

loaded_features = loaded_meta["features"]
today_date = "2026-02-10"
target_term = "여름계절학기"
term_months = get_term_months(target_term)

smoke_df["RUL_개월"] = predict_assets(loaded_model, smoke_df[loaded_features], smoke_df["운용연차"])
smoke_df["AI예측고장일"] = smoke_df["RUL_개월"].apply(lambda x: calculate_failure_date(x, today_date))
smoke_df["포트폴리오_영역"] = smoke_df.apply(lambda r: assign_matrix_zone(r["RUL_개월"], r["장비중요도"]), axis=1)
smoke_df["고장예상여부"] = smoke_df["RUL_개월"].apply(lambda x: classify_term_failure(x, term_months))

smoke_summary = {
    "prediction_rows": len(smoke_df),
    "term_failure_count": int(smoke_df["고장예상여부"].sum()),
    "min_rul_months": float(smoke_df["RUL_개월"].min()),
    "max_rul_months": float(smoke_df["RUL_개월"].max()),
}
display(pd.DataFrame([smoke_summary]))
display(smoke_df[["물품고유번호", "G2B목록명", "RUL_개월", "AI예측고장일", "포트폴리오_영역", "고장예상여부"]].head())

## 8. 배포 경로 교체 셀

아래 셀은 기본값이 `False`라서 실행해도 기존 서버 호환 모델을 덮어쓰지 않는다. 최종 모델을 확정한 뒤에만 `DEPLOY_TO_SERVER_COMPAT_PATH = True`로 바꿔 실행한다.

In [ ]:
DEPLOY_TO_SERVER_COMPAT_PATH = False

if DEPLOY_TO_SERVER_COMPAT_PATH:
    deploy_dir = PROJECT_ROOT / "ai_model" / "saved_models" / "random_forest"
    deploy_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(best_model, deploy_dir / "rf_final_model.pkl")
    with open(deploy_dir / "model_meta.json", "w", encoding="utf-8") as f:
        json.dump(model_meta, f, ensure_ascii=False, indent=2)
    print(f"배포 호환 경로 저장 완료: {deploy_dir}")
else:
    print("배포 경로는 변경하지 않았습니다. 최종 확정 후 True로 바꿔 실행하세요.")